# QuantLib Benchmark: SOFR + TermSOFR3m + CLP ICP Curves & Swap Sensitivities

Replicates the `QuantSupport` sensitivity example using QuantLib-Python.

**Key conventions (matching quantsupport):**
- Reference date: 2025-11-11
- Settlement: T+2 calendar days → 2025-11-13 (NullCalendar)
- Log-linear interpolation on discount factors (flat-forward)
- **NullCalendar** / **Unadjusted** BDC everywhere
- **Backward** date generation, no end-of-month
- Floating leg uses **simple forward rates** per coupon period (NOT daily-compounded OIS)
- Fixed leg: Semiannual, Floating leg: Quarterly
- Day count: Act/360, Compounding: Simple
- Bootstrap instruments start at reference_date (not T+2)
- DV01 via central-difference bump ±1bp

In [41]:
import QuantLib as ql
import pandas as pd
import numpy as np
import json
import os

from data_loading import load_rust_results, qs_sens_dict, qs_npv, qs_cashflows_df, qs_curve_df, qs_sens_full
from display import display_dv01, compare_discount_factors, display_cashflows
from ql_helpers import make_schedule, make_vanilla_swap, compute_dv01

## 1. Global Settings

In [42]:
reference_date = ql.Date(11, 11, 2025)
ql.Settings.instance().evaluationDate = reference_date

# quantsupport uses NullCalendar + Unadjusted for everything
calendar = ql.NullCalendar()
day_count = ql.Actual360()

# T+2 settlement (calendar days, no holiday adjustment)
start_date = reference_date + 2  # 2025-11-13
notional = 10_000_000.0

print(f"Reference date : {reference_date}")
print(f"Start date     : {start_date}")

Reference date : November 11th, 2025
Start date     : November 13th, 2025


## 1b. Load Rust (quantsupport) Results

The sensitivity example outputs a JSON file with NPV, sensitivities (raw exposure + DV01),
cashflow details, and curve discount factors.  We load it here and use it throughout the
notebook instead of hard-coded values.

In [43]:
rust_products, rust_curves = load_rust_results()

print(f"Loaded Rust results: {len(rust_products)} products, {len(rust_curves)} curves")
for label, prod in rust_products.items():
    print(f"  - {label}: NPV = {prod['npv']:.2f}, {len(prod['sensitivities'])} sensitivities, {len(prod['cashflows'])} cashflows")

Loaded Rust results: 4 products, 4 curves
  - SOFR OIS 5Y Swap: NPV = 342.60, 13 sensitivities, 34 cashflows
  - TermSOFR3m 5Y Swap: NPV = 88754.68, 25 sensitivities, 34 cashflows
  - CLP ICP OIS 5Y Swap (USD collateral): NPV = 288.27, 21 sensitivities, 34 cashflows
  - Cross-Currency Swap CLP/USD 5Y (receive CLP ICP, pay USD SOFR): NPV = 156003.53, 34 sensitivities, 44 cashflows


## 2. Market Data

All quotes from `examples/sensitivity/data/quotes.json`.

In [44]:
# SOFR OIS curve quotes
sofr_quotes = {
    "FixedRateDeposit_USD_SOFR_1D": 0.04330,
    "OIS_USD_SOFR_3M":  0.04335,
    "OIS_USD_SOFR_6M":  0.04250,
    "OIS_USD_SOFR_1Y":  0.04100,
    "OIS_USD_SOFR_2Y":  0.03920,
    "OIS_USD_SOFR_3Y":  0.03840,
    "OIS_USD_SOFR_4Y":  0.03800,
    "OIS_USD_SOFR_5Y":  0.03780,
    "OIS_USD_SOFR_7Y":  0.03770,
    "OIS_USD_SOFR_10Y": 0.03790,
    "OIS_USD_SOFR_15Y": 0.03850,
    "OIS_USD_SOFR_20Y": 0.03880,
    "OIS_USD_SOFR_30Y": 0.03750,
}

# TermSOFR3m curve quotes (basis spreads over SOFR)
termsofr_quotes = {
    "FixedRateDeposit_USD_TermSOFR3m_3M": 0.04365,
    "BasisSwap_USD_SOFR_TermSOFR3m_6M":  0.000127237986,
    "BasisSwap_USD_SOFR_TermSOFR3m_1Y":  0.000160745572,
    "BasisSwap_USD_SOFR_TermSOFR3m_2Y":  0.000196298404,
    "BasisSwap_USD_SOFR_TermSOFR3m_3Y":  0.000224136825,
    "BasisSwap_USD_SOFR_TermSOFR3m_4Y":  0.000243421966,
    "BasisSwap_USD_SOFR_TermSOFR3m_5Y":  0.000262792898,
    "BasisSwap_USD_SOFR_TermSOFR3m_7Y":  0.000298893256,
    "BasisSwap_USD_SOFR_TermSOFR3m_10Y": 0.000335114864,
    "BasisSwap_USD_SOFR_TermSOFR3m_15Y": 0.000378100224,
    "BasisSwap_USD_SOFR_TermSOFR3m_20Y": 0.000415419903,
    "BasisSwap_USD_SOFR_TermSOFR3m_30Y": 0.000465137190,
}

# CLP ICP curve quotes
icp_quotes = {
    "FixedRateDeposit_CLP_ICP_1D":  0.0500,
    "FixedRateDeposit_CLP_ICP_3M":  0.0498,
    "OIS_CLP_ICP_6M":  0.0495,
    "OIS_CLP_ICP_1Y":  0.0485,
    "OIS_CLP_ICP_2Y":  0.0470,
    "OIS_CLP_ICP_3Y":  0.0455,
    "OIS_CLP_ICP_5Y":  0.0440,
    "OIS_CLP_ICP_7Y":  0.0430,
    "OIS_CLP_ICP_10Y": 0.0425,
    "OIS_CLP_ICP_20Y": 0.0420,
}

# Merge all quotes into a single handle store for bumping
all_quote_handles = {}
for quotes_dict in [sofr_quotes, termsofr_quotes, icp_quotes]:
    for name, rate in quotes_dict.items():
        all_quote_handles[name] = ql.SimpleQuote(rate)

def make_handle(name):
    return ql.QuoteHandle(all_quote_handles[name])

print(f"Total quotes: {len(all_quote_handles)}")

Total quotes: 35


## 3. Helper Functions

Build swaps matching quantsupport conventions: simple forward rates (VanillaSwap),
NullCalendar, Unadjusted, Backward date generation.

In [45]:
def make_handle(name):
    return ql.QuoteHandle(all_quote_handles[name])

## 4. Bootstrap the SOFR OIS Curve

quantsupport bootstraps OIS swaps with:
- Fixed Semi / Float Quarterly, both starting at reference_date
- Simple forward rates (not daily-compounded OIS)
- NPV = 0 as the bootstrap objective

We use `SwapRateHelper` with a quarterly `IborIndex` (simple forwards), not `OISRateHelper`.

In [46]:
sofr_helpers = []

# 1D deposit
sofr_helpers.append(
    ql.DepositRateHelper(
        make_handle("FixedRateDeposit_USD_SOFR_1D"),
        ql.Period(1, ql.Days),
        0,                    # fixing days = 0 (starts at ref date)
        calendar,             # NullCalendar
        ql.Unadjusted,
        False,
        day_count,
    )
)

# IborIndex for projecting simple forwards in bootstrap
sofr_ibor_for_bootstrap = ql.IborIndex(
    "SimpleSOFR", ql.Period(3, ql.Months),
    0, ql.USDCurrency(), calendar, ql.Unadjusted, False, day_count,
)

ois_tenors = [
    ("OIS_USD_SOFR_3M",  ql.Period(3, ql.Months)),
    ("OIS_USD_SOFR_6M",  ql.Period(6, ql.Months)),
    ("OIS_USD_SOFR_1Y",  ql.Period(1, ql.Years)),
    ("OIS_USD_SOFR_2Y",  ql.Period(2, ql.Years)),
    ("OIS_USD_SOFR_3Y",  ql.Period(3, ql.Years)),
    ("OIS_USD_SOFR_4Y",  ql.Period(4, ql.Years)),
    ("OIS_USD_SOFR_5Y",  ql.Period(5, ql.Years)),
    ("OIS_USD_SOFR_7Y",  ql.Period(7, ql.Years)),
    ("OIS_USD_SOFR_10Y", ql.Period(10, ql.Years)),
    ("OIS_USD_SOFR_15Y", ql.Period(15, ql.Years)),
    ("OIS_USD_SOFR_20Y", ql.Period(20, ql.Years)),
    ("OIS_USD_SOFR_30Y", ql.Period(30, ql.Years)),
]

for name, tenor in ois_tenors:
    sofr_helpers.append(
        ql.SwapRateHelper(
            make_handle(name),
            tenor,
            calendar,
            ql.Semiannual,
            ql.Unadjusted,
            day_count,
            sofr_ibor_for_bootstrap,
        )
    )

sofr_curve = ql.PiecewiseLogLinearDiscount(reference_date, sofr_helpers, day_count)
sofr_curve.enableExtrapolation()
sofr_curve_handle = ql.YieldTermStructureHandle(sofr_curve)

print("SOFR curve bootstrapped.")
for dt, df in sofr_curve.nodes():
    yf = day_count.yearFraction(reference_date, dt)
    print(f"  {dt}  DF={df:.10f}  YF={yf:.6f}")

SOFR curve bootstrapped.
  November 11th, 2025  DF=1.0000000000  YF=0.000000
  November 12th, 2025  DF=0.9998797367  YF=0.002778
  February 11th, 2026  DF=0.9890430514  YF=0.255556
  May 11th, 2026  DF=0.9790789858  YF=0.502778
  November 11th, 2026  DF=0.9597061980  YF=1.013889
  November 11th, 2027  DF=0.9243882594  YF=2.027778
  November 11th, 2028  DF=0.8908288784  YF=3.044444
  November 11th, 2029  DF=0.8585791637  YF=4.058333
  November 11th, 2030  DF=0.8273197252  YF=5.072222
  November 11th, 2032  DF=0.7673403029  YF=7.102778
  November 11th, 2035  DF=0.6833842370  YF=10.144444
  November 11th, 2040  DF=0.5586557094  YF=15.219444
  November 11th, 2045  DF=0.4566471726  YF=20.291667
  November 11th, 2055  DF=0.3281415350  YF=30.436111


### SOFR Curve — Discount Factor Comparison

In [47]:
compare_discount_factors(sofr_curve, "SOFR", rust_curves, reference_date, day_count, "SOFR OIS Curve")


────────────────────────────────────────────────────────────────────────────────
  Discount Factor Comparison: SOFR OIS Curve
────────────────────────────────────────────────────────────────────────────────
  Date               YF            QL DF            QS DF           Diff
  --------------------------------------------------------------------
  2025-11-12     0.0028     0.9998797367     0.9998797367       1.11e-16
  2025-12-11     0.0833     0.9964134592     0.9964134592       1.11e-16
  2026-02-11     0.2556     0.9890430514     0.9890430514       1.11e-16
  2026-05-11     0.5028     0.9790789858     0.9790789858      -6.88e-15
  2026-11-11     1.0139     0.9597061980     0.9597061980      -7.57e-14
  2027-11-11     2.0278     0.9243882594     0.9243882594       2.00e-15
  2028-11-11     3.0444     0.8908288784     0.8908288784      -4.76e-14
  2029-11-11     4.0583     0.8585791637     0.8585791637       2.22e-15
  2030-11-11     5.0722     0.8273197252     0.8273197252       

## 5. Price SOFR 5Y Swap & Sensitivities

In [48]:
maturity_5y = start_date + ql.Period(5, ql.Years)
print(f"Swap: {start_date} → {maturity_5y}, fixed = 3.78%, notional = {notional:,.0f}")

sofr_swap = make_vanilla_swap(
    start_date, maturity_5y, 0.0378, notional,
    sofr_curve_handle, sofr_curve_handle,
    calendar, day_count,
)

# Read quantsupport results from JSON
rs_sofr_dv01 = qs_sens_full(rust_products, "SOFR OIS 5Y Swap")
rs_sofr_npv = qs_npv(rust_products, "SOFR OIS 5Y Swap")

sofr_handles = {k: all_quote_handles[k] for k in sofr_quotes}
sofr_npv, sofr_dv01 = compute_dv01(sofr_swap, sofr_handles)
display_dv01("SOFR OIS 5Y Swap", sofr_npv, sofr_dv01, rs_sofr_dv01, rs_npv=rs_sofr_npv)

Swap: November 13th, 2025 → November 13th, 2030, fixed = 3.78%, notional = 10,000,000
═══════════════════════════════════════════════════════════════════════════════════════════════
  SOFR OIS 5Y Swap
  NPV =         342.60 USD  (quantsupport:         342.60 USD)
═══════════════════════════════════════════════════════════════════════════════════════════════
  Pillar                                           QL DV01    QS DV01    QS Exposure       Diff
  -------------------------------------------------------------------------------------------
  FixedRateDeposit_USD_SOFR_1D                        2.75       2.75     27462.6312       0.00
  OIS_USD_SOFR_3M                                     2.78       2.78     27768.6580      -0.00
  OIS_USD_SOFR_6M                                     0.10       0.10      1014.6112       0.00
  OIS_USD_SOFR_1Y                                     0.00       0.00        30.0393       0.00
  OIS_USD_SOFR_2Y                                    -0.00      -0

### SOFR OIS 5Y Swap — Cashflow Details (quantsupport)

In [49]:
display_cashflows(rust_products, "SOFR OIS 5Y Swap")

payment_date      cashflow_type    side currency          notional rate_pct  year_fraction         amount  discount_factor
  2026-05-13    FixedRateCoupon Receive      USD 10,000,000.000000  3.7800%       0.502778 190,050.000000         0.978866
  2026-11-13    FixedRateCoupon Receive      USD 10,000,000.000000  3.7800%       0.511111 193,200.000000         0.959509
  2027-05-13    FixedRateCoupon Receive      USD 10,000,000.000000  3.7800%       0.502778 190,050.000000         0.941833
  2027-11-13    FixedRateCoupon Receive      USD 10,000,000.000000  3.7800%       0.511111 193,200.000000         0.924201
  2028-05-13    FixedRateCoupon Receive      USD 10,000,000.000000  3.7800%       0.505556 191,100.000000         0.907362
  2028-11-13    FixedRateCoupon Receive      USD 10,000,000.000000  3.7800%       0.511111 193,200.000000         0.890649
  2029-05-13    FixedRateCoupon Receive      USD 10,000,000.000000  3.7800%       0.502778 190,050.000000         0.874511
  2029-11-13    

## 6. Bootstrap TermSOFR3m Curve

The TermSOFR3m curve is built from:
- A 3M deposit (outright rate)
- Basis swaps: pay SOFR + spread, receive TermSOFR3m Q → NPV = 0

The basis spread quote means `TermSOFR3m_par_swap_rate ≈ SOFR_par_swap_rate + spread`.

In [50]:
termsofr_helpers = []

# 3M deposit
termsofr_helpers.append(
    ql.DepositRateHelper(
        make_handle("FixedRateDeposit_USD_TermSOFR3m_3M"),
        ql.Period(3, ql.Months),
        0, calendar, ql.Unadjusted, False, day_count,
    )
)

# For each basis swap tenor, compute SOFR par rate + spread → TermSOFR3m par rate
sofr_ibor_linked = ql.IborIndex(
    "SimpleSOFR", ql.Period(3, ql.Months),
    0, ql.USDCurrency(), calendar, ql.Unadjusted, False, day_count,
    sofr_curve_handle,
)

termsofr_ibor_for_bootstrap = ql.IborIndex(
    "TermSOFR3m", ql.Period(3, ql.Months),
    0, ql.USDCurrency(), calendar, ql.Unadjusted, False, day_count,
)

basis_tenors = [
    ("BasisSwap_USD_SOFR_TermSOFR3m_6M",  ql.Period(6, ql.Months)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_1Y",  ql.Period(1, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_2Y",  ql.Period(2, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_3Y",  ql.Period(3, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_4Y",  ql.Period(4, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_5Y",  ql.Period(5, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_7Y",  ql.Period(7, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_10Y", ql.Period(10, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_15Y", ql.Period(15, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_20Y", ql.Period(20, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_30Y", ql.Period(30, ql.Years)),
]

# Store par rate SimpleQuote objects so bumping basis spreads propagates
# to the TermSOFR3m curve during DV01 computation.
termsofr_par_quotes = {}

# Annuity ratios: the basis spread enters the quarterly floating leg,
# but SwapRateHelper uses a semiannual fixed-rate swap. The exact
# conversion is:  r_TermSOFR = r_SOFR + spread × A_qtr / A_semi
# where A_qtr / A_semi is the ratio of floating-to-fixed leg annuities.
annuity_ratios = {}

for name, tenor in basis_tenors:
    maturity = reference_date + tenor
    fixed_sched = make_schedule(reference_date, maturity, ql.Period(ql.Semiannual), calendar)
    float_sched = make_schedule(reference_date, maturity, ql.Period(ql.Quarterly), calendar)

    # Get SOFR par rate for this tenor
    temp_swap = ql.VanillaSwap(
        ql.VanillaSwap.Receiver, 1.0,
        fixed_sched, 0.0, day_count,
        float_sched, sofr_ibor_linked, 0.0, day_count,
    )
    temp_swap.setPricingEngine(ql.DiscountingSwapEngine(sofr_curve_handle))
    sofr_par = temp_swap.fairRate()

    # Annuity ratio: floating-leg BPS / fixed-leg BPS
    # BPS = Σ τ_i × N × DF_i × 1e-4, so the ratio cancels units.
    ratio = abs(temp_swap.floatingLegBPS()) / abs(temp_swap.fixedLegBPS())
    annuity_ratios[name] = ratio

    basis_spread = all_quote_handles[name].value()
    termsofr_par = sofr_par + basis_spread * ratio

    par_sq = ql.SimpleQuote(termsofr_par)
    termsofr_par_quotes[name] = par_sq

    termsofr_helpers.append(
        ql.SwapRateHelper(
            ql.QuoteHandle(par_sq),
            tenor,
            calendar,
            ql.Semiannual,
            ql.Unadjusted,
            day_count,
            termsofr_ibor_for_bootstrap,
        )
    )

termsofr_curve = ql.PiecewiseLogLinearDiscount(reference_date, termsofr_helpers, day_count)
termsofr_curve.enableExtrapolation()
termsofr_curve_handle = ql.YieldTermStructureHandle(termsofr_curve)

print("TermSOFR3m curve bootstrapped.")
for dt, df in termsofr_curve.nodes():
    yf = day_count.yearFraction(reference_date, dt)
    print(f"  {dt}  DF={df:.10f}  YF={yf:.6f}")
print()
print("Annuity ratios (A_qtr / A_semi):")
for name, r in annuity_ratios.items():
    print(f"  {name}: {r:.6f}")

TermSOFR3m curve bootstrapped.
  November 11th, 2025  DF=1.0000000000  YF=0.000000
  February 11th, 2026  DF=0.9889680613  YF=0.255556
  May 11th, 2026  DF=0.9790173488  YF=0.502778
  November 11th, 2026  DF=0.9595519369  YF=1.013889
  November 11th, 2027  DF=0.9240245706  YF=2.027778
  November 11th, 2028  DF=0.8902264182  YF=3.044444
  November 11th, 2029  DF=0.8577367613  YF=4.058333
  November 11th, 2030  DF=0.8262210133  YF=5.072222
  November 11th, 2032  DF=0.7657054656  YF=7.102778
  November 11th, 2035  DF=0.6810318318  YF=10.144444
  November 11th, 2040  DF=0.5553449820  YF=15.219444
  November 11th, 2045  DF=0.4526013223  YF=20.291667
  November 11th, 2055  DF=0.3232702248  YF=30.436111

Annuity ratios (A_qtr / A_semi):
  BasisSwap_USD_SOFR_TermSOFR3m_6M: 1.005173
  BasisSwap_USD_SOFR_TermSOFR3m_1Y: 1.005097
  BasisSwap_USD_SOFR_TermSOFR3m_2Y: 1.004906
  BasisSwap_USD_SOFR_TermSOFR3m_3Y: 1.004822
  BasisSwap_USD_SOFR_TermSOFR3m_4Y: 1.004776
  BasisSwap_USD_SOFR_TermSOFR3m_5Y:

### TermSOFR3m Curve — Discount Factor Comparison

In [51]:
compare_discount_factors(termsofr_curve, "TermSOFR3m", rust_curves, reference_date, day_count, "TermSOFR3m Curve")


────────────────────────────────────────────────────────────────────────────────
  Discount Factor Comparison: TermSOFR3m Curve
────────────────────────────────────────────────────────────────────────────────
  Date               YF            QL DF            QS DF           Diff
  --------------------------------------------------------------------
  2025-11-12     0.0028     0.9998794286     0.9998794286       0.00e+00
  2025-12-11     0.0833     0.9963891733     0.9963891733       1.11e-16
  2026-02-11     0.2556     0.9889680613     0.9889680613       1.11e-16
  2026-05-11     0.5028     0.9790173488     0.9790174893      -1.41e-07
  2026-11-11     1.0139     0.9595519369     0.9595515340       4.03e-07
  2027-11-11     2.0278     0.9240245706     0.9240227911       1.78e-06
  2028-11-11     3.0444     0.8902264182     0.8902229721       3.45e-06
  2029-11-11     4.0583     0.8577367613     0.8577316334       5.13e-06
  2030-11-11     5.0722     0.8262210133     0.8262142303     

## 7. Price TermSOFR3m 5Y Swap & Sensitivities

Receive fixed 4.00% semi vs pay TermSOFR3m quarterly, 10MM USD.
Discount with SOFR (CSA = SOFR/USD).

In [52]:
termsofr_swap = make_vanilla_swap(
    start_date, maturity_5y, 0.04, notional,
    termsofr_curve_handle, sofr_curve_handle,  # forecast=TermSOFR3m, discount=SOFR
    calendar, day_count,
    ibor_name="TermSOFR3m",
)

print(f"TermSOFR3m swap: {start_date} → {maturity_5y}, fixed = 4.00%")
print(f"NPV = {termsofr_swap.NPV():,.2f} USD")
print(f"Fair rate = {termsofr_swap.fairRate():.6%}")

TermSOFR3m swap: November 13th, 2025 → November 13th, 2030, fixed = 4.00%
NPV = 88,828.25 USD
Fair rate = 3.805514%


In [53]:
# Read quantsupport results from JSON
rs_termsofr_dv01 = qs_sens_full(rust_products, "TermSOFR3m 5Y Swap")
rs_termsofr_npv = qs_npv(rust_products, "TermSOFR3m 5Y Swap")

# Custom DV01 for TermSOFR3m: when bumping a basis spread quote by Δ,
# the par rate changes by Δ × (A_qtr / A_semi) — the annuity ratio
# accounts for the spread entering the quarterly leg but the par rate
# being a semiannual fixed rate.
def compute_termsofr_dv01(swap, handles_to_bump, par_quotes, a_ratios):
    bump = 1e-4
    base = swap.NPV()
    results = {}
    for name, sq in handles_to_bump.items():
        orig = sq.value()
        par_sq = par_quotes.get(name)
        par_orig = par_sq.value() if par_sq else None
        ratio = a_ratios.get(name, 1.0)

        sq.setValue(orig + bump)
        if par_sq:
            par_sq.setValue(par_orig + bump * ratio)
        up = swap.NPV()

        sq.setValue(orig - bump)
        if par_sq:
            par_sq.setValue(par_orig - bump * ratio)
        dn = swap.NPV()

        sq.setValue(orig)
        if par_sq:
            par_sq.setValue(par_orig)
        results[name] = (up - dn) / 2.0
    return base, results

combined_handles = {}
for k in termsofr_quotes:
    combined_handles[k] = all_quote_handles[k]
for k in sofr_quotes:
    combined_handles[k] = all_quote_handles[k]

termsofr_npv, termsofr_dv01 = compute_termsofr_dv01(
    termsofr_swap, combined_handles, termsofr_par_quotes, annuity_ratios,
)
display_dv01("TermSOFR3m 5Y Swap", termsofr_npv, termsofr_dv01, rs_termsofr_dv01,
             rs_npv=rs_termsofr_npv)

═══════════════════════════════════════════════════════════════════════════════════════════════
  TermSOFR3m 5Y Swap
  NPV =       88828.25 USD  (quantsupport:       88754.68 USD)
═══════════════════════════════════════════════════════════════════════════════════════════════
  Pillar                                           QL DV01    QS DV01    QS Exposure       Diff
  -------------------------------------------------------------------------------------------
  FixedRateDeposit_USD_TermSOFR3m_3M                  2.77       5.56     55605.4795      -2.79
  BasisSwap_USD_SOFR_TermSOFR3m_6M                    1.88      -0.01      -111.6338       1.90
  BasisSwap_USD_SOFR_TermSOFR3m_1Y                   -0.30       0.00        20.6891      -0.30
  BasisSwap_USD_SOFR_TermSOFR3m_2Y                    1.65      -0.00       -42.7949       1.65
  BasisSwap_USD_SOFR_TermSOFR3m_3Y                    3.69      -0.00       -22.2111       3.70
  BasisSwap_USD_SOFR_TermSOFR3m_4Y                    

### TermSOFR3m 5Y Swap — Cashflow Details (quantsupport)

In [54]:
display_cashflows(rust_products, "TermSOFR3m 5Y Swap")

payment_date      cashflow_type    side currency          notional rate_pct  year_fraction         amount  discount_factor
  2026-05-13    FixedRateCoupon Receive      USD 10,000,000.000000  4.0000%       0.502778 201,111.111111         0.978866
  2026-11-13    FixedRateCoupon Receive      USD 10,000,000.000000  4.0000%       0.511111 204,444.444444         0.959509
  2027-05-13    FixedRateCoupon Receive      USD 10,000,000.000000  4.0000%       0.502778 201,111.111111         0.941833
  2027-11-13    FixedRateCoupon Receive      USD 10,000,000.000000  4.0000%       0.511111 204,444.444444         0.924201
  2028-05-13    FixedRateCoupon Receive      USD 10,000,000.000000  4.0000%       0.505556 202,222.222222         0.907362
  2028-11-13    FixedRateCoupon Receive      USD 10,000,000.000000  4.0000%       0.511111 204,444.444444         0.890649
  2029-05-13    FixedRateCoupon Receive      USD 10,000,000.000000  4.0000%       0.502778 201,111.111111         0.874511
  2029-11-13    

## 8. Bootstrap CLP ICP Curve

Same conventions: NullCalendar, Unadjusted, Backward, Semi fixed / Quarterly float.

In [55]:
icp_helpers = []

# 1D deposit
icp_helpers.append(
    ql.DepositRateHelper(
        make_handle("FixedRateDeposit_CLP_ICP_1D"),
        ql.Period(1, ql.Days),
        0, calendar, ql.Unadjusted, False, day_count,
    )
)

# 3M deposit
icp_helpers.append(
    ql.DepositRateHelper(
        make_handle("FixedRateDeposit_CLP_ICP_3M"),
        ql.Period(3, ql.Months),
        0, calendar, ql.Unadjusted, False, day_count,
    )
)

icp_ibor_for_bootstrap = ql.IborIndex(
    "SimpleICP", ql.Period(3, ql.Months),
    0, ql.CLPCurrency(), calendar, ql.Unadjusted, False, day_count,
)

icp_tenors = [
    ("OIS_CLP_ICP_6M",  ql.Period(6, ql.Months)),
    ("OIS_CLP_ICP_1Y",  ql.Period(1, ql.Years)),
    ("OIS_CLP_ICP_2Y",  ql.Period(2, ql.Years)),
    ("OIS_CLP_ICP_3Y",  ql.Period(3, ql.Years)),
    ("OIS_CLP_ICP_5Y",  ql.Period(5, ql.Years)),
    ("OIS_CLP_ICP_7Y",  ql.Period(7, ql.Years)),
    ("OIS_CLP_ICP_10Y", ql.Period(10, ql.Years)),
    ("OIS_CLP_ICP_20Y", ql.Period(20, ql.Years)),
]

for name, tenor in icp_tenors:
    icp_helpers.append(
        ql.SwapRateHelper(
            make_handle(name),
            tenor,
            calendar,
            ql.Semiannual,
            ql.Unadjusted,
            day_count,
            icp_ibor_for_bootstrap,
        )
    )

icp_curve = ql.PiecewiseLogLinearDiscount(reference_date, icp_helpers, day_count)
icp_curve.enableExtrapolation()
icp_curve_handle = ql.YieldTermStructureHandle(icp_curve)

print("CLP ICP curve bootstrapped.")
for dt, df in icp_curve.nodes():
    yf = day_count.yearFraction(reference_date, dt)
    print(f"  {dt}  DF={df:.10f}  YF={yf:.6f}")

CLP ICP curve bootstrapped.
  November 11th, 2025  DF=1.0000000000  YF=0.000000
  November 12th, 2025  DF=0.9998611304  YF=0.002778
  February 11th, 2026  DF=0.9874332660  YF=0.255556
  May 11th, 2026  DF=0.9757168470  YF=0.502778
  November 11th, 2026  DF=0.9525936769  YF=1.013889
  November 11th, 2027  DF=0.9101874364  YF=2.027778
  November 11th, 2028  DF=0.8722701847  YF=3.044444
  November 11th, 2030  DF=0.8025817025  YF=5.072222
  November 11th, 2032  DF=0.7404184262  YF=7.102778
  November 11th, 2035  DF=0.6543257930  YF=10.144444
  November 11th, 2045  DF=0.4324258301  YF=20.291667


### CLP ICP Curve — Discount Factor Comparison

In [56]:
compare_discount_factors(icp_curve, "ICP", rust_curves, reference_date, day_count, "CLP ICP Curve")


────────────────────────────────────────────────────────────────────────────────
  Discount Factor Comparison: CLP ICP Curve
────────────────────────────────────────────────────────────────────────────────
  Date               YF            QL DF            QS DF           Diff
  --------------------------------------------------------------------
  2025-11-12     0.0028     0.9998611304     0.9998611304       0.00e+00
  2025-12-11     0.0833     0.9958837145     0.9958837145       0.00e+00
  2026-02-11     0.2556     0.9874332660     0.9874332660       0.00e+00
  2026-05-11     0.5028     0.9757168470     0.9757932516      -7.64e-05
  2026-11-11     1.0139     0.9525936769     0.9526192657      -2.56e-05
  2027-11-11     2.0278     0.9101874364     0.9102091081      -2.17e-05
  2028-11-11     3.0444     0.8722701847     0.8722857253      -1.55e-05
  2029-11-11     4.0583     0.8367007171     0.8367291342      -2.84e-05
  2030-11-11     5.0722     0.8025817025     0.8026219204      -4

## 9. Price CLP ICP 5Y Swap & Sensitivities

Receive fixed 4.40% semi vs pay ICP quarterly, 5B CLP.

**Discount curve**: In quantsupport, CLP cashflows are discounted off `Collateral(CLP, USD)` —
a cross-currency basis curve built from FX forwards + XCcy swaps.
Here we use the ICP curve itself (self-discount) as an approximation,
so NPVs and DV01s will differ from quantsupport.
NPVs converted to USD at FX spot = 935.

In [57]:
clp_notional = 5_000_000_000.0
fx_spot = 935.0

icp_swap = make_vanilla_swap(
    start_date, maturity_5y, 0.0440, clp_notional,
    icp_curve_handle, icp_curve_handle,  # self-discount
    calendar, day_count,
    ibor_name="SimpleICP", ccy=ql.CLPCurrency(),
)

icp_npv_clp = icp_swap.NPV()
icp_npv_usd = icp_npv_clp / fx_spot

print(f"CLP ICP swap: {start_date} → {maturity_5y}, fixed = 4.40%")
print(f"NPV = {icp_npv_clp:,.2f} CLP = {icp_npv_usd:,.2f} USD")
print(f"Fair rate = {icp_swap.fairRate():.6%}")

CLP ICP swap: November 13th, 2025 → November 13th, 2030, fixed = 4.40%
NPV = 263,033.98 CLP = 281.32 USD
Fair rate = 4.398827%


In [58]:
# Read quantsupport results from JSON
rs_icp_dv01 = qs_sens_full(rust_products, "CLP ICP OIS 5Y Swap (USD collateral)")
rs_icp_npv = qs_npv(rust_products, "CLP ICP OIS 5Y Swap (USD collateral)")

icp_handles = {k: all_quote_handles[k] for k in icp_quotes}
icp_npv_usd_base, icp_dv01 = compute_dv01(icp_swap, icp_handles, fx_divisor=fx_spot)

print("Note: quantsupport discounts CLP flows with Collateral(CLP,USD) curve;")
print("      here we use ICP self-discount, so some difference is expected.\n")
display_dv01("CLP ICP OIS 5Y Swap (USD equiv)", icp_npv_usd_base, icp_dv01, rs_icp_dv01,
             rs_npv=rs_icp_npv)

Note: quantsupport discounts CLP flows with Collateral(CLP,USD) curve;
      here we use ICP self-discount, so some difference is expected.

═══════════════════════════════════════════════════════════════════════════════════════════════
  CLP ICP OIS 5Y Swap (USD equiv)
  NPV =         281.32 USD  (quantsupport:         288.27 USD)
═══════════════════════════════════════════════════════════════════════════════════════════════
  Pillar                                           QL DV01    QS DV01    QS Exposure       Diff
  -------------------------------------------------------------------------------------------
  FixedRateDeposit_CLP_ICP_1D                         1.47       1.46     14624.0843       0.01
  FixedRateDeposit_CLP_ICP_3M                         1.48       1.56     15647.9710      -0.08
  OIS_CLP_ICP_6M                                      0.06      -0.07      -745.1355       0.14
  OIS_CLP_ICP_1Y                                      0.00       0.03       257.7173      -0

### CLP ICP OIS 5Y Swap — Cashflow Details (quantsupport)

In [59]:
display_cashflows(rust_products, "CLP ICP OIS 5Y Swap (USD collateral)")

payment_date      cashflow_type    side currency             notional rate_pct  year_fraction             amount  discount_factor
  2026-05-13    FixedRateCoupon Receive      CLP 5,000,000,000.000000  4.4000%       0.502778 110,611,111.111111         0.965688
  2026-11-13    FixedRateCoupon Receive      CLP 5,000,000,000.000000  4.4000%       0.511111 112,444,444.444444         0.950219
  2027-05-13    FixedRateCoupon Receive      CLP 5,000,000,000.000000  4.4000%       0.502778 110,611,111.111111         0.928791
  2027-11-13    FixedRateCoupon Receive      CLP 5,000,000,000.000000  4.4000%       0.511111 112,444,444.444444         0.907521
  2028-05-13    FixedRateCoupon Receive      CLP 5,000,000,000.000000  4.4000%       0.505556 111,222,222.222222         0.888545
  2028-11-13    FixedRateCoupon Receive      CLP 5,000,000,000.000000  4.4000%       0.511111 112,444,444.444444         0.869758
  2029-05-13    FixedRateCoupon Receive      CLP 5,000,000,000.000000  4.4000%       0.502

## 10. Summary

In [60]:
print("NPV Comparison Summary")
print("=" * 70)
print(f"{'Product':<40} {'QuantLib':>12} {'quantsupport':>12}")
print("-" * 70)
print(f"{'SOFR OIS 5Y Swap (USD)':<40} {sofr_npv:>12.2f} {qs_npv(rust_products, 'SOFR OIS 5Y Swap'):>12.2f}")
print(f"{'TermSOFR3m 5Y Swap (USD)':<40} {termsofr_npv:>12.2f} {qs_npv(rust_products, 'TermSOFR3m 5Y Swap'):>12.2f}")
print(f"{'CLP ICP 5Y Swap (USD equiv)':<40} {icp_npv_usd_base:>12.2f} {qs_npv(rust_products, 'CLP ICP OIS 5Y Swap (USD collateral)'):>12.2f}")
print("=" * 70)
print()
print("Convention notes:")
print("  - VanillaSwap with IborIndex (simple fwd rates), not OvernightIndexedSwap")
print("  - NullCalendar, Unadjusted BDC, Backward date generation")
print("  - Log-linear on DFs (flat forward between nodes)")
print("  - TermSOFR3m: static par-rate approach for basis curve")
print("  - CLP ICP: self-discount in QL (missing Collateral(CLP,USD) curve)")

NPV Comparison Summary
Product                                      QuantLib quantsupport
----------------------------------------------------------------------
SOFR OIS 5Y Swap (USD)                         342.60       342.60
TermSOFR3m 5Y Swap (USD)                     88828.25     88754.68
CLP ICP 5Y Swap (USD equiv)                    281.32       288.27

Convention notes:
  - VanillaSwap with IborIndex (simple fwd rates), not OvernightIndexedSwap
  - NullCalendar, Unadjusted BDC, Backward date generation
  - Log-linear on DFs (flat forward between nodes)
  - TermSOFR3m: static par-rate approach for basis curve
  - CLP ICP: self-discount in QL (missing Collateral(CLP,USD) curve)
